In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('data/main_stats_1778352449.csv')

# Calculate the difference between perceived error and actual hardware error
df['error_diff'] = df['err_ms'] - df['mean_hw_delay_ms']

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Perceived Error vs Actual Error
sns.scatterplot(data=df, x='err_ms', y='mean_hw_delay_ms', ax=axes[0, 0], alpha=0.5)
axes[0, 0].set_title('Perceived Error vs Actual HW Error')
axes[0, 0].set_xlabel('Perceived Error (err_ms)')
axes[0, 0].set_ylabel('Actual HW Error (mean_hw_delay_ms)')

# Add a y=x reference line
min_val = min(df['err_ms'].min(), df['mean_hw_delay_ms'].min())
max_val = max(df['err_ms'].max(), df['mean_hw_delay_ms'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', label='y=x (Perfect Match)')
axes[0, 0].legend()

# 2. Error Difference vs Delta Up
sns.scatterplot(data=df, x='delta_up_ms', y='error_diff', ax=axes[0, 1], alpha=0.5)
axes[0, 1].set_title('Error Discrepancy vs Delta Up')
axes[0, 1].set_xlabel('Delta Up (ms)')
axes[0, 1].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

# 3. Error Difference vs Delta Down
sns.scatterplot(data=df, x='delta_down_ms', y='error_diff', ax=axes[1, 0], alpha=0.5)
axes[1, 0].set_title('Error Discrepancy vs Delta Down')
axes[1, 0].set_xlabel('Delta Down (ms)')
axes[1, 0].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

# 4. Error Difference vs Delay
sns.scatterplot(data=df, x='delay_ms', y='error_diff', ax=axes[1, 1], alpha=0.5)
axes[1, 1].set_title('Error Discrepancy vs Total Delay')
axes[1, 1].set_xlabel('Delay (ms)')
axes[1, 1].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

plt.tight_layout()
# plt.savefig('error_analysis.png')


In [28]:
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor

# Load the data
df = pd.read_csv('data/main_stats_1778523494.csv')

# ---------------------------------------------------------
# 1. Covariance Matrix
# ---------------------------------------------------------
cols_of_interest = ['err_ms', 'mean_hw_delay_ms', 'delta_up_ms', 'delta_down_ms']
cov_matrix = df[cols_of_interest].cov()
print("--- Covariance Matrix ---")
print(cov_matrix)
print("\n")

# ---------------------------------------------------------
# 2. OLS Regression (Predicting HW Error)
# ---------------------------------------------------------
# We drop delta_down_ms from X because err_ms == -delta_down_ms (perfect multicollinearity)
X = df[['err_ms', 'delta_up_ms']]
X = sm.add_constant(X) # Adds the y-intercept (constant)
y = df['mean_hw_delay_ms']

model = sm.OLS(y, X).fit()
print("--- OLS Regression Summary ---")
print(model.summary())
print("\n")

# ---------------------------------------------------------
# 3. Random Forest Regressor (Feature Importance)
# ---------------------------------------------------------
rf = RandomForestRegressor(random_state=42)
rf.fit(df[['err_ms', 'delta_up_ms']], df['mean_hw_delay_ms'])

print("--- Random Forest Feature Importances ---")
print(f"err_ms (perceived error): {rf.feature_importances_[0]:.4f}")
print(f"delta_up_ms (uplink delay): {rf.feature_importances_[1]:.4f}")


In [29]:
import pandas as pd
import glob

def load_and_weight_datasets(file_pattern="data/*.csv", min_entries=10):
    """
    Loads multiple CSV files, filters out small ones, and assigns weights 
    based on observation index.
    
    Args:
        file_pattern (str): The path pattern to find your CSVs (e.g., 'data/*.csv' or a list of files).
        min_entries (int): Minimum number of rows required to keep the dataset.
        
    Returns:
        pd.DataFrame: A combined dataframe with a new 'sample_weight' column.
    """
    # If a string is provided, find all matching files. Otherwise, assume it's a list.
    if isinstance(file_pattern, str):
        file_paths = glob.glob(file_pattern)
    else:
        file_paths = file_pattern
        
    valid_dfs = []
    
    for file in file_paths:
        try:
            df = pd.read_csv(file)
            # Filter out any big errors
            df = df[df['err_ms'] <= 100]
            df = df[df['delta_up_ms'] <= 100]
            df = df[df['delta_down_ms'] <= 100]
            # Check condition: amount of entries must be greater than min_entries (10)
            if len(df) > min_entries:
                
                # Weighting strategy: Weight according to the index.
                # We add 1 so the first row (index 0) gets a weight of 1, 
                # index 66 gets a weight of 67, etc.
                df['sample_weight'] = np.sqrt(len(df) )
                
                valid_dfs.append(df)
            else:
                print(f"Skipping {file}: Only {len(df)} entries (needs > {min_entries}).")
                
        except Exception as e:
            print(f"Error reading {file}: {e}")
            
    # Combine all valid dataframes into one master dataframe
    if not valid_dfs:
        print("Warning: No valid dataframes found matching the criteria.")
        return pd.DataFrame()
        
    combined_df = pd.concat(valid_dfs, ignore_index=True)
    return combined_df

# --- Example Usage ---
df_combined = load_and_weight_datasets('data/main_stats_*.csv')


In [30]:
# Prepare features and target
import numpy as np
df_combined = df_combined.replace([np.inf, -np.inf], np.nan).dropna(subset=cols_of_interest)
X = df_combined[['err_ms', 'delta_up_ms','delta_down_ms']]
X = sm.add_constant(X)
y = df_combined['mean_hw_delay_ms']
weights = df_combined['sample_weight']

# Use WLS (Weighted Least Squares) instead of OLS
model = sm.WLS(y, X, weights=weights).fit()
print(model.summary())


In [31]:
df_combined['error_diff'] = df_combined['err_ms'] - df_combined['mean_hw_delay_ms']

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Perceived Error vs Actual Error
sns.scatterplot(data=df_combined, x='err_ms', y='mean_hw_delay_ms', ax=axes[0, 0], alpha=0.5)
axes[0, 0].set_title('Perceived Error vs Actual HW Error')
axes[0, 0].set_xlabel('Perceived Error (err_ms)')
axes[0, 0].set_ylabel('Actual HW Error (mean_hw_delay_ms)')

# Add a y=x reference line
min_val = min(df_combined['err_ms'].min(), df['mean_hw_delay_ms'].min())
max_val = max(df_combined['err_ms'].max(), df['mean_hw_delay_ms'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', label='y=x (Perfect Match)')
axes[0, 0].legend()

# 2. Error Difference vs Delta Up
sns.scatterplot(data=df_combined, x='delta_up_ms', y='error_diff', ax=axes[0, 1], alpha=0.5)
axes[0, 1].set_title('Error Discrepancy vs Delta Up')
axes[0, 1].set_xlabel('Delta Up (ms)')
axes[0, 1].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

# 3. Error Difference vs Delta Down
sns.scatterplot(data=df_combined, x='delta_down_ms', y='error_diff', ax=axes[1, 0], alpha=0.5)
axes[1, 0].set_title('Error Discrepancy vs Delta Down')
axes[1, 0].set_xlabel('Delta Down (ms)')
axes[1, 0].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

# 4. Error Difference vs Delay
sns.scatterplot(data=df_combined, x='delay_ms', y='error_diff', ax=axes[1, 1], alpha=0.5)
axes[1, 1].set_title('Error Discrepancy vs Total Delay')
axes[1, 1].set_xlabel('Delay (ms)')
axes[1, 1].set_ylabel('Discrepancy (err_ms - mean_hw_delay_ms)')

plt.tight_layout()
#


In [40]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Prepare features
X = df_combined[['err_ms', 'delta_up_ms', 'delta_down_ms']]
y = df_combined['mean_hw_delay_ms']
weights = df_combined['sample_weight']

rf = RandomForestRegressor(random_state=42)

# Pass the weights into the fit function
rf.fit(X, y, sample_weight=weights)

print("--- Weighted Random Forest Feature Importances ---")
print(f"err_ms (perceived error): {rf.feature_importances_[0]:.4f}")
print(f"delta_up_ms (uplink delay): {rf.feature_importances_[1]:.4f}")
predictions = rf.predict(X)

# Calculate the Weighted R-squared
r2 = r2_score(y, predictions, sample_weight=weights)
print(f"Random Forest R-squared: {r2:.3f}")


In [42]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
df = pd.read_csv('data/main_stats_1778523494.csv')
X_train = df[['err_ms', 'delta_up_ms', 'delta_down_ms']]
y_train = df['mean_hw_delay_ms']
kernel = RBF() + WhiteKernel()
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5)
gp.fit(X_train, y_train)
y_pred, y_std = gp.predict(X, return_std=True)
